In [1]:
import pandas as pd
from datetime import datetime
from tqdm.auto import tqdm

# Import RAG

In [2]:
import sys
sys.path.append('../scripts')
import rag
import vectors

# Loading data

## Answers to synthetic questions

In [3]:
df_synth_a = pd.read_csv('../data/data-synth-answer.csv', sep='\t', dtype=str)
df_synth_a

,pmid,ollama_seed,answer_llama3.2:1b,answer_gemma3:1b
0,40247608,0,"Based on the provided context, a significant p...","Okay, based on the provided context and the re..."
1,40247608,1,Based on the provided context and papers from ...,"Okay, based on the provided text, here’s the a..."
2,40247608,2,Based on the provided context and papers from ...,"Okay, let's analyze the provided text and answ..."
3,40247608,3,Based on the context provided by the papers fr...,"Okay, based on the provided text and the PubMe..."
4,40247608,4,"Based on the provided context, which includes ...","Okay, let's analyze the provided context and a..."
5,40267907,0,This text appears to be a collection of scient...,"Okay, here's an analysis of the provided text,..."
6,40267907,1,"Based on the provided context, which includes ...","Okay, here's an answer based on the provided c..."
7,40267907,2,Based on the provided context and papers from ...,"Okay, based on the provided text and the quest..."
8,40267907,3,The primary difference between individuals wit...,"Okay, let’s analyze the provided text and answ..."
9,40267907,4,"Based on the provided PubMed articles, I found...","Okay, let’s analyze these papers based on the ..."


## Synthetic questions

In [4]:
df_synth_q = pd.read_csv('../data/data-synth-question.csv', sep='\t', dtype=str)
df_synth_q

,pmid,ollama_seed,synthetic_question
0,40247608,0,What causes a significant portion of individua...
1,40247608,1,Is Fragile X syndrome considered a type of aut...
2,40247608,2,What causes some individuals with autism to ex...
3,40247608,3,What causes the unique communication styles an...
4,40247608,4,What are the main differences between Fragile ...
...,...,...,...
495,40933686,0,What challenges do autistic individuals face w...
496,40933686,1,Can we do away with the stigma surrounding aut...
497,40933686,2,What strategies do universities need to implem...
498,40933686,3,What challenges do autistic individuals face w...


## Abstracts

In [5]:
kb_records = rag.get_kb_records('../data/data-kb.csv')
df_abstr = pd.DataFrame.from_records(kb_records)[['pmid', 'abstract']]
df_abstr

,pmid,abstract
0,40939192,Due to the boom in the use of certain psychede...
1,40938792,The self-presentation of psychiatric disorders...
2,40938690,Autism spectrum disorder (ASD) is often comorb...
3,40938348,
4,40938167,Stereotypies currently occupy an important pla...
...,...,...
2995,40246257,Young children with autism spectrum disorder (...
2996,40245451,The talker's mouth provides significant multim...
2997,40245419,
2998,40245385,An individual education plan (IEP) is a key el...


## Put together

In [6]:
df_together = df_synth_a.copy()
df_together = df_together.merge(df_synth_q, on=['pmid', 'ollama_seed'], how='left')
df_together = df_together.merge(df_abstr, on=['pmid'], how='left')
df_together

,pmid,ollama_seed,answer_llama3.2:1b,answer_gemma3:1b,synthetic_question,abstract
0,40247608,0,"Based on the provided context, a significant p...","Okay, based on the provided context and the re...",What causes a significant portion of individua...,Fragile X syndrome is the most common inherite...
1,40247608,1,Based on the provided context and papers from ...,"Okay, based on the provided text, here’s the a...",Is Fragile X syndrome considered a type of aut...,Fragile X syndrome is the most common inherite...
2,40247608,2,Based on the provided context and papers from ...,"Okay, let's analyze the provided text and answ...",What causes some individuals with autism to ex...,Fragile X syndrome is the most common inherite...
3,40247608,3,Based on the context provided by the papers fr...,"Okay, based on the provided text and the PubMe...",What causes the unique communication styles an...,Fragile X syndrome is the most common inherite...
4,40247608,4,"Based on the provided context, which includes ...","Okay, let's analyze the provided context and a...",What are the main differences between Fragile ...,Fragile X syndrome is the most common inherite...
5,40267907,0,This text appears to be a collection of scient...,"Okay, here's an analysis of the provided text,...",What is the underlying mechanism by which bi-a...,Congenital disorders of glycosylation (CDGs) c...
6,40267907,1,"Based on the provided context, which includes ...","Okay, here's an answer based on the provided c...",How is autism diagnosed?,Congenital disorders of glycosylation (CDGs) c...
7,40267907,2,Based on the provided context and papers from ...,"Okay, based on the provided text and the quest...",What are the main differences between autism a...,Congenital disorders of glycosylation (CDGs) c...
8,40267907,3,The primary difference between individuals wit...,"Okay, let’s analyze the provided text and answ...",What is the primary difference between individ...,Congenital disorders of glycosylation (CDGs) c...
9,40267907,4,"Based on the provided PubMed articles, I found...","Okay, let’s analyze these papers based on the ...",What is the underlying cause of congenital dis...,Congenital disorders of glycosylation (CDGs) c...


# Evaluation by cosine similarity

## Embed texts

In [7]:
# vectorizer handle
print(vectors.model_handle)

multi-qa-MiniLM-L6-cos-v1


In [8]:
# vector representations
print(datetime.now())
column_names_to_vectorize = ['answer_llama3.2:1b', 'answer_gemma3:1b', 'abstract']
vectorized = {name : vectors.model.encode(df_together[name].to_list()) \
for name in column_names_to_vectorize}
print(datetime.now())

2025-09-13 03:01:05.616817
2025-09-13 03:01:10.471186


## Compute cosine similarities

In [9]:
def get_cosine_similarities(model_handle):
    similarities = []
    for i in range(len(vectorized['abstract'])):
        vector_answer = vectorized['answer_'+model_handle][i]
        vector_abstract = vectorized['abstract'][i]
        similarity = vectors.model.similarity(vector_answer, vector_abstract).item()
        similarities.append(similarity)
    return similarities

In [10]:
# get similarities by model handle
df_similarities = pd.DataFrame({handle : get_cosine_similarities(handle) \
for handle in ['llama3.2:1b', 'gemma3:1b']})
df_similarities

,llama3.2:1b,gemma3:1b
0,0.756706,0.761975
1,0.754927,0.706509
2,0.498336,0.490033
3,0.475071,0.509501
4,0.681909,0.690602
5,0.404637,0.456378
6,0.175539,0.236917
7,0.246631,0.279420
8,0.739152,0.723981
9,0.884359,0.695862


## Compare models

In [11]:
# descriptive statistics, for llama3.2
df_similarities['llama3.2:1b'].describe()

count    30.000000
mean      0.601798
std       0.179742
min       0.175539
25%       0.485288
50%       0.616384
75%       0.753245
max       0.884359
Name: llama3.2:1b, dtype: float64

In [12]:
# descriptive statistics, for llama3.2
df_similarities['gemma3:1b'].describe()

count    30.000000
mean      0.581255
std       0.172831
min       0.225641
25%       0.494900
50%       0.572926
75%       0.719613
max       0.834518
Name: gemma3:1b, dtype: float64

In [13]:
# descriptive statistics, difference
(df_similarities['gemma3:1b']-df_similarities['llama3.2:1b']).describe()

count    30.000000
mean     -0.020543
std       0.074329
min      -0.188497
25%      -0.067938
50%      -0.013352
75%       0.026449
max       0.185838
dtype: float64

In [14]:
print(datetime.now())

2025-09-13 03:01:10.528904
